# 14 · SkewerJAX and fake_spectra: how each stage scales

Notebook 13 times the two codes on one target. This notebook moves one parameter at a time around it and follows every
stage of the four entries (fake_spectra; SkewerJAX default and matched on the CPU; SkewerJAX default on the GPU), with the
calls and the protocol of notebook 13:

- **A, the simulation resolution:** TNG50-4, TNG50-3 and TNG50-2 (snapshot 25, $z = 3$) at $N_{\max}^2 = 128^2$ skewers;
- **B, the number of skewers:** TNG50-2, $n = 4, 16, 32, 64, 128, 256$ skewers per side (16 to 65,536 skewers);
- **C1, the pixel width:** TNG50-2, $128^2$ skewers, pixels of 1, 2, 4, 8 and 16 km/s;
- **C2, the sampling of the midpoint option:** the earlier default cut every cell every 0.25 km/s and evaluated the
  profile at the middle of each piece; the default now integrates it over each crossing and has no sampling. C2 measures
  what the option loses at each spacing, against the default.

The skewers run along $y$ on the nested grid $x, z = iL/n$, so the **reference skewer** $(L/4, L/4)$, notebook 12's,
belongs to every grid: its flux, stored for every entry and point, is the precision check of every block.

**By default the notebook reads the records** of `scripts/benchmarks/benchmark_scaling.py` (`benchmarks/results/scaling/`,
one JSON per point: stage times, JIT time, page-cache state, peak memory, the machine, the agreement over all skewers and
the flux of the reference skewer) and runs in about a minute. **`RUN = True` re-runs the benchmark** for the points
without a record (resumable; each point in its own process): about 2.3 hours on this machine for all of them.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")             # this kernel reads the records and plots; the benchmark workers set their own
import inspect, json, logging
logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import (PALETTE_CYCLE, SCALING_ENTRY_STYLES, add_panel_letters, plot_flux_checks,
                                      plot_scaling_block, plot_stage_grid, use_backend, use_style)
from skewerjax.utils.paths import BENCHMARK_RECORDS_DIR, DATA_DIR
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()        # paper style: Tol's colour-blind-safe "bright" colours and colour maps, minimum font sizes

## The calls

Every point runs the four entries with these calls (`skewerjax.comparison.scaling`), the calls of notebook 13:
fake_spectra on every skewer at once, timed from outside, once per pixel grid; SkewerJAX with **one pass over the
snapshot for all its entries and grids** (`stream.stream_cells`, both physics configurations), then the tiles of each
entry on each grid (`stream.tiles_optical_depth`). **Cold**: the snapshot is dropped from the page cache before each code
reads it. The warm-up runs the same entries on TNG50-4 first (the JIT compilation, `jit_s`, not in the bars). At $n \le
16$, where the reads vary most, each code runs twice and the faster run is kept.

In [ ]:
from skewerjax.comparison import scaling as S
print(inspect.getsource(S.run_fake_spectra))
print(inspect.getsource(S.skewerjax_point))

## The records

`RUN = False` (the default) reads the records; `True` measures every point without one, then reads them all (the same as
`python scripts/benchmarks/benchmark_scaling.py`). `N_MAX` is that of notebook 13.

In [ ]:
RUN = False                                    # True: run the benchmark for the points without a record (resumes)
N_MAX = S.N_MAX                                # the grid of notebook 13 and of blocks A and C1
OUT = BENCHMARK_RECORDS_DIR / "scaling"        # one JSON per point ($SKEWERJAX_BENCHMARK_RESULTS)
POINTS = S.scaling_points(N_MAX)
if RUN:
    S.run_points(POINTS, OUT, data_dir=DATA_DIR)
R = S.load_records(OUT)
C2 = R.pop("C2", None)                         # the precision study of block C2 (benchmark_scaling.py --c2)
print(f"{len(R)} of {len(POINTS)} points recorded in {OUT}" + ("; C2 recorded" if C2 else "; C2 not recorded"))
for c in POINTS:
    r = R.get(c["id"])
    if r is None:
        print(f"  {c['id']:32s} ({', '.join(c['blocks'])}): not recorded")
        continue
    mem = max(x["memory"]["rss_GB"] for x in r["entries"].values() if x.get("memory"))
    print(f"  {c['id']:32s} ({', '.join(c['blocks']):8s}) {r['date']}, page cache before each read: "
          + ", ".join(f"{v:.0%}" for v in r["cache"].values()) + f"; JIT {r.get('jit_s', 0):.0f} s; peak memory {mem:.0f} GB; wall {r['wall_s']:.0f} s")
m = next(iter(R.values()))["machine"] if R else {}
print(f"machine: {m.get('cpu')}, {m.get('logical_cpus')} logical CPUs, {m.get('ram_GB', float('nan')):.0f} GB RAM; GPU {next(iter(R.values()))['gpu'] if R else '-'}")

In [ ]:
ENTRIES = list(S.ENTRIES)
XLABEL = {"A": "gas cells in the snapshot", "B": "skewers", "C1": "pixel width [km/s]"}


def have(b):
    """Whether block b has records to show (a partial RUN = True leaves some blocks empty: they are skipped)."""
    ok = len(S.block_rows(R, b, N_MAX)) > 0
    if not ok:
        print(f"block {b}: no record yet (RUN = True measures it)")
    return ok


def stage_figure(b):
    rows = S.block_rows(R, b, N_MAX)
    fig, axs = plot_stage_grid([r["stages"] for r in rows], labels=[r["title"] for r in rows], jit=[r["jit"] for r in rows])
    add_panel_letters(axs, labels=[f"{chr(97 + i)}{j + 1}" for i in range(len(rows)) for j in range(3)])
    return fig, axs


def scaling_figure(b):
    rows = S.block_rows(R, b, N_MAX)
    fig, axs = plot_scaling_block([r["x"] for r in rows], [r["stages"] for r in rows], xlabel=XLABEL[b])
    add_panel_letters(axs)
    return fig, axs


def print_stats(stats):
    """The rms and max |dF| of every curve of a precision figure, over every pixel."""
    for (col, lab, line), (r, m) in stats.items():
        print(f"{col:42s} {lab:26s} {line:6s} rms {r:.1e}  max {m:.1e}")


def agreement_table(b):
    """rms and max dF over every pixel of every skewer: the matched mode and the default against fake_spectra, the GPU
    against the CPU."""
    for r in S.block_rows(R, b, N_MAX):
        print(f"{r['title']:34s} " + "; ".join(f"{k}: rms {v['rms_dF']:.1e} max {v['max_dF']:.1e}" for k, v in r["agreement"].items()))


def totals_table(b):
    for r in S.block_rows(R, b, N_MAX):
        t = {e: sum(st.values()) for e, st in r["stages"].items()}
        print(f"{r['title']:34s} " + "; ".join(f"{e} {v:.0f} s" + ("" if e == S.FS else f" ({t[S.FS] / v:.2f}x)") for e, v in t.items()))

## A · The simulation resolution

TNG50-4, -3 and -2 hold 19.3 million, 152 million and 1.2 billion gas cells in boxes of the same size: the same skewers
cross 8 times more, smaller cells from one to the next.

**Where the time goes, point by point** (as notebook 13, one row per point: (a1) the stages, (a2) their shares, (a3) the totals and the speed-ups over fake_spectra; then (b1), ...).

In [ ]:
if have("A"):
    fig, axs = stage_figure("A")
    fig.suptitle("fake_spectra's time follows the snapshot; SkewerJAX's, the cells its skewers cross", fontsize=21)
    plt.show()

In [ ]:
if have("A"):
    fig, axs = scaling_figure("A")
    fig.suptitle("From TNG50-4 to TNG50-2 fake_spectra slows 16×, SkewerJAX 4–10×", fontsize=21)
    plt.show()
    totals_table("A")
    agreement_table("A")

**Precision.** The flux of the reference skewer from every entry (top), and $|\Delta F|$ against fake_spectra for
$\mathrm{Ly}\alpha$ (middle) and Si III (bottom), one column per snapshot; the rms and the max over its pixels are printed
below the figure. This skewer has almost no Si III ($|\Delta F| < 10^{-6}$): the agreement over every skewer, printed
above, covers Si III. The GPU's line lies on the CPU's.

In [ ]:
if have("A"):
    cols = []
    for r in S.block_rows(R, "A", N_MAX):
        fl, v = r["flux"], (np.arange(r["nbins"]) + 0.5) * r["dv_kms"]
        ref = fl[S.FS]
        cols.append(dict(title=r["title"], v=v, style=SCALING_ENTRY_STYLES, F={e: fl[e]["HI-LyA"] for e in ENTRIES if e in fl},
                         dF={e: (np.subtract(fl[e]["HI-LyA"], ref["HI-LyA"]), np.subtract(fl[e]["SiIII-1206"], ref["SiIII-1206"]))
                             for e in ENTRIES if e in fl and e != S.FS}))
    fig, axs, stats = plot_flux_checks(cols)
    add_panel_letters(axs)
    fig.suptitle("The matched mode is fake_spectra to 1e-6 at every resolution; the default differs by its own physics", fontsize=21)
    plt.show()
    print_stats(stats)

## B · The number of skewers

TNG50-2, from $4^2$ to $256^2$ skewers.

**Where the time goes, point by point** (as notebook 13, one row per point: (a1) the stages, (a2) their shares, (a3) the totals and the speed-ups over fake_spectra; then (b1), ...).

In [ ]:
if have("B"):
    fig, axs = stage_figure("B")
    fig.suptitle("Up to 4,096 skewers SkewerJAX's time is mostly its one read of the snapshot; from 16,384 on, its τ sum grows to most of it", fontsize=21)
    plt.show()

In [ ]:
if have("B"):
    fig, axs = scaling_figure("B")
    fig.suptitle("SkewerJAX is faster than fake_spectra at every number of skewers, 16 to 65,536", fontsize=21)
    plt.show()
    totals_table("B")
    agreement_table("B")

**Precision.** Each entry's reference skewer at every $n$ against its own result at the largest $n$: a skewer must
not depend on how many others are computed with it (top: the flux, below: $|\Delta F|$).

In [ ]:
if have("B"):
    rows = S.block_rows(R, "B", N_MAX)
    last = rows[-1]
    v = (np.arange(last["nbins"]) + 0.5) * last["dv_kms"]
    cols = []
    for e in ENTRIES:
        if e not in last["flux"]:
            continue
        ref = last["flux"][e]
        styles, F, dF = {}, {}, {}
        for k, r in enumerate(rows):
            fl = r["flux"].get(e)
            if fl is None:
                continue
            lab = f"n = {r['record']['config']['n']}"
            styles[lab] = dict(color=PALETTE_CYCLE[k % len(PALETTE_CYCLE)], lw=2.0)
            F[lab] = fl["HI-LyA"]
            if r is not last:
                dF[lab] = (np.subtract(fl["HI-LyA"], ref["HI-LyA"]), np.subtract(fl["SiIII-1206"], ref["SiIII-1206"]))
        cols.append(dict(title=f"{e}: against n = {last['record']['config']['n']}", v=v, F=F, dF=dF, style=styles))
    fig, axs, stats = plot_flux_checks(cols)
    add_panel_letters(axs)
    fig.suptitle("The reference skewer does not depend on the grid around it", fontsize=21)
    plt.show()
    print_stats(stats)

## C1 · The pixel width

TNG50-2, $128^2$ skewers, pixels of 1 to 16 km/s. SkewerJAX's pass and its crossings are shared by the five grids: only
the $\tau$ sum sees the pixels (the matched mode runs on the 1 km/s grid only). fake_spectra runs once per grid. The
precision check compares every width with the 1 km/s result **rebinned to that width: the average of $F$ over each coarse
pixel** (each 1 km/s pixel weighted by its overlap with it, `scaling.rebin_average`).

**Where the time goes, point by point** (as notebook 13, one row per point: (a1) the stages, (a2) their shares, (a3) the totals and the speed-ups over fake_spectra; then (b1), ...).

In [ ]:
if have("C1"):
    fig, axs = stage_figure("C1")
    fig.suptitle("Coarser pixels shorten SkewerJAX's τ sum; its pass, 80 s, stays", fontsize=21)
    plt.show()

In [ ]:
if have("C1"):
    fig, axs = scaling_figure("C1")
    fig.suptitle("From 1 to 16 km/s pixels the CPU's τ sum falls 2.3×; the GPU's was already short", fontsize=21)
    plt.show()
    totals_table("C1")
    agreement_table("C1")

**Precision.** The flux of the reference skewer at each pixel width (top), and $|\Delta F|$ of every entry against its
own 1 km/s flux averaged over the coarse pixels: the difference between computing on coarse pixels and averaging fine ones.

In [ ]:
if have("C1"):
    rows = S.block_rows(R, "C1", N_MAX)
    fine = rows[0]                                          # 1 km/s
    e1 = np.arange(fine["nbins"] + 1) * fine["dv_kms"]      # the edges of the 1 km/s pixels [km/s]
    cols = []
    for r in rows[1:]:
        ec = np.arange(r["nbins"] + 1) * r["dv_kms"]
        v = 0.5 * (ec[1:] + ec[:-1])
        fl = r["flux"]
        reb = {e: {ln: S.rebin_average(fine["flux"][e][ln], e1, ec) for ln in S.LINE_IDS} for e in ENTRIES if e in fl}
        cols.append(dict(title=r["title"], v=v, style=SCALING_ENTRY_STYLES, F={e: fl[e]["HI-LyA"] for e in ENTRIES if e in fl},
                         dF={e: (np.subtract(fl[e]["HI-LyA"], reb[e]["HI-LyA"]), np.subtract(fl[e]["SiIII-1206"], reb[e]["SiIII-1206"]))
                             for e in ENTRIES if e in fl}))
    fig, axs, stats = plot_flux_checks(cols)
    add_panel_letters(axs)
    fig.suptitle("Against the 1 km/s flux averaged over the coarse pixels, every entry alike", fontsize=21)
    plt.show()
    print_stats(stats)

## C2 · The sampling of the midpoint option

The default integrates the profile over each cell crossing (`profile_integration="integrated"`): it has no sampling
along the skewer. The option `profile_integration="midpoint"` on the dual-segment pieces (`deposition="voronoi_segments"`)
is the earlier default: the pieces are cut at the ends of the dual segments of points spaced every 0.125 to 4 km/s, and
each piece absorbs with the profile at its line centre. Block C2 (`scaling.run_c2`, float64, $8^2$ TNG50-2 skewers)
compares it with the default at every spacing: rms and max $\Delta F$ over every pixel, the pieces per skewer and the
$\tau$ time per skewer.

In [ ]:
if C2 is None:
    print("block C2: no record yet (python scripts/benchmarks/benchmark_scaling.py --c2)")
else:
    sp = sorted(C2["midpoint"], key=float)
    x = np.array([float(s) for s in sp])
    fig, axs = plt.subplots(1, 2, figsize=(18, 6.5), layout="constrained")
    for j, (ln, lab) in enumerate(zip(S.LINE_IDS, ("Lyα", "Si III"))):
        c = PALETTE_CYCLE[j]
        axs[0].loglog(x, [C2["midpoint"][s][f"{ln}:rms_dF"] for s in sp], "o-", color=c, lw=2.5, label=f"{lab}, rms")
        axs[0].loglog(x, [C2["midpoint"][s][f"{ln}:max_dF"] for s in sp], "s--", color=c, lw=2.0, label=f"{lab}, max")
    axs[0].set(xlabel="point spacing [km/s]", ylabel=r"$|\Delta F|$ against the integrated profile")
    axs[0].legend()
    axs[1].loglog(x, [1e3 * C2["midpoint"][s]["tau_s_per_skewer"] for s in sp], "o-", color=PALETTE_CYCLE[2], lw=2.5, label="midpoint, sampled pieces")
    axs[1].axhline(1e3 * C2["integrated"]["tau_s_per_skewer"], color=PALETTE_CYCLE[3], lw=2.5, ls=":", label="integrated, cell crossings (default)")
    ax2 = axs[1].twinx()
    ax2.loglog(x, [C2["midpoint"][s]["pieces_per_skewer"] for s in sp], "^--", color="0.4", lw=1.8, label="pieces per skewer")
    ax2.set_ylabel("pieces per skewer")
    axs[1].set(xlabel="point spacing [km/s]", ylabel="τ time per skewer [ms]")
    axs[1].legend(loc="upper right")
    add_panel_letters(axs)
    fig.suptitle("The midpoint sums converge as the spacing squared: at 0.125 km/s, 3× the integrated profile's τ time", fontsize=21)
    plt.show()
    for s in sp:
        m = C2["midpoint"][s]
        print(f"{s:>6s} km/s: {m['pieces_per_skewer']:7.0f} pieces per skewer, tau {1e3 * m['tau_s_per_skewer']:6.2f} ms per skewer; "
              + "; ".join(f"{ln} rms {m[f'{ln}:rms_dF']:.1e} max {m[f'{ln}:max_dF']:.1e}" for ln in S.LINE_IDS))
    print(f"integrated (default): tau {1e3 * C2['integrated']['tau_s_per_skewer']:.2f} ms per skewer; precision {C2['precision']}")

## Discussion

**Every target is met** (TNG50-2, cold, 1 km/s, float32). SkewerJAX's three entries are faster than fake_spectra at
every number of skewers: 12 times faster at 16 skewers (25.7 s against 308 s on the CPU), 5.5 times at 1,024, 3.7 times at $128^2$ (134.6 s;
GPU 105.1 s, matched 180.1 s) and 2.8 times at $256^2$ (385.5 s; GPU 273.0 s, matched 562.0 s), with a peak memory of 83 GB.
Before this work (commit `01d9900`) the CPU took 28.5 s, 328 s and about 3,000 s at 16, 1,024 and $128^2$ skewers.

- **A.** fake_spectra reads and converts every cell, so its time follows the snapshot (30.6 s, 87.3 s, 498 s). SkewerJAX
  reads the snapshot once near the skewers and then pays per crossing: on TNG50-4 its default CPU entry is just faster
  (25.3 s), its GPU 3 times faster, and its matched mode slower (42.4 s: fake_spectra's per-cell walk on 16,384 skewers
  against a fixed cost of 6 s).
- **B.** fake_spectra costs ~310 s at any N (I/O 166 s, gas physics 126 s), plus 11.6 ms per skewer. SkewerJAX's
  pass grows from 27 s (16 skewers) to 55 s (1,024), when it reads every page of the snapshot, then with the cells it
  keeps (81 s at $128^2$, 161 s at $256^2$); its $\tau$ sum adds 3.7 ms per skewer on the CPU, 1.9 ms on the GPU and 6.3 ms
  in the matched mode. The reference skewer does not depend on the grid around it.
- **C1.** Only SkewerJAX's $\tau$ sum sees the pixels: from 1 to 16 km/s it falls from 58 s to 25 s on the CPU, and the
  pass (80 s) remains. fake_spectra's time does not fall with coarser pixels (435–586 s).
- **C2.** The midpoint option converges as the square of its spacing ($\mathrm{Ly}\alpha$ rms $\Delta F$ 3.3e-6 at 0.25 km/s, the earlier
  default, 8.6e-7 at 0.125 km/s) and costs 1.7–3.2 times the integrated profile's $\tau$ time at those spacings: the default has
  no sampling to choose.

The agreement over every skewer: the matched mode against fake_spectra, $\mathrm{Ly}\alpha$ rms $\Delta F$ $\le$ 2.0e-7 at every point; the GPU
against the CPU, rms $\le$ 7.6e-7. `docs/performance.md` explains the design behind these numbers and its limits.
